# Efecto de capacitación sobre ingresos: preparación

Consulta la [metodología](../docs/metodologia.md) y ejecuta las etapas en orden. Los comentarios se calculan con los resultados guardados.

In [1]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
def _buscar_proyecto(paquete):
    """Encuentra el paquete propio desde el proyecto, tema o portafolio."""
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "src" / paquete).is_dir():
            return base
        for container in (base, base / "Analisis"):
            for pattern in (f"*/src/{paquete}", f"*/*/src/{paquete}"):
                for source in sorted(container.glob(pattern)):
                    parts = source.relative_to(container).parts
                    if "back" in parts or any(p.startswith(".") for p in parts):
                        continue
                    candidate = source.parent.parent
                    if (candidate / "pyproject.toml").is_file():
                        return candidate
    return None

PROJECT_ROOT = _buscar_proyecto("training_program_impact")
if PROJECT_ROOT is None:
    raise RuntimeError("Abre este notebook desde su proyecto independiente.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from training_program_impact.alerts import configurar_notebook, mostrar_alertas
from IPython.display import display, Markdown, Image
configurar_notebook()


In [2]:
from training_program_impact.analysis import run_stage
import pandas as pd
metrics,comment=run_stage(PROJECT_ROOT,"preparacion")
display(Markdown(comment))
scalars={k:v for k,v in metrics.items() if not isinstance(v,(dict,list))}
if scalars:display(pd.DataFrame(list(scalars.items()),columns=["Indicador","Valor"]))
if "candidatos" in metrics:display(pd.DataFrame(metrics["candidatos"]))
for filename in ["eda.png","evaluacion.png"]:
    path=PROJECT_ROOT/"reports/preparacion"/filename
    if path.exists():display(Image(filename=str(path)))
for filename in ["estimaciones.csv","metricas_test.csv","horas_por_empleado.csv"]:
    path=PROJECT_ROOT/"reports/preparacion"/filename
    if path.exists():display(pd.read_csv(path).head(20))

# Efecto de capacitación sobre ingresos: preparacion

Se conservan **445 participantes**, **185 asignados al programa** y **260 controles**, sin deduplicar participantes por atributos. Resultado: ingresos de 1978 en USD nominales. Solo se incluyen covariables previas. Es una submuestra experimental histórica; no se mezclan controles observacionales ni se crean splits predictivos. El estimando es el efecto de **asignación al programa en esta muestra**, no el efecto entre quienes completaron la capacitación.


,Indicador,Valor
0,fuente,https://users.nber.org/~rdehejia/data/.nswdata...
1,sha256,e5a3a38aba34088094d817b7a3939bcfdb2d280474373a...
2,filas,445
3,asignados_programa,185
4,controles,260
5,resultado,re78: ingresos anuales de 1978 en USD nominales
6,muestra,Submuestra experimental NSW Dehejia–Wahba; no ...
7,particion_predictiva,No aplica: estimación causal sobre la muestra ...


In [3]:
mostrar_alertas(PROJECT_ROOT / "reports/alertas/01_preparacion_datos.json")
from training_program_impact.informe import generar_informe
generar_informe(PROJECT_ROOT)
display(Markdown("[Abrir informe integral](../reports/informe_final/informe_final.md)"))

[Abrir informe integral](../reports/informe_final/informe_final.md)